# Semana 26 · ¿Por qué el cielo es azul y el Sol se pone rojo?

Este cuaderno acompaña a la sesión de proyecto de la semana 26: exponenciales, masa de aire y el color del Sol a lo largo de la tarde.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. Potencias frente a exponenciales

$x^4$ y $2^x$ se parecen al escribirlas, pero se comportan de forma muy distinta. Las dibujamos juntas en escala normal y en escala logarítmica en el eje vertical.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(0.1, 25, 300)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
for ax in (a1, a2):
    ax.plot(x, x**4, label="potencia $x^4$")
    ax.plot(x, 2**x, label="exponencial $2^x$")
    ax.set_xlabel("x"); ax.legend(); ax.grid(alpha=0.3)
a1.set_ylim(0, 3e5); a1.set_title("Escala normal")
a2.set_yscale("log"); a2.set_title("Eje vertical logarítmico")
plt.tight_layout(); plt.show()

En el eje logarítmico, la exponencial es una **recta**: cada unidad de $x$ multiplica el valor por 2, es decir, suma siempre lo mismo a su logaritmo. La potencia, en cambio, se curva y se aplana.

## 2. La luz que atraviesa el aire

Con $I = I_0 e^{-kx}$ y un coeficiente $k$, dibujamos cuánta luz queda, y comprobamos que $\ln I$ es una recta.

In [ ]:
k = 0.105                      # por km (el 90 % pasa cada km)
x = np.linspace(0, 30, 200)
I = 1.0 * np.exp(-k * x)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
a1.plot(x, I); a1.set_xlabel("km de aire"); a1.set_ylabel("fracción de luz que queda"); a1.grid(alpha=0.3)
a1.axhline(0.5, color="gray", ls="--"); a1.axvline(np.log(2) / k, color="gray", ls="--")
a1.set_title(f"Espesor mitad: {np.log(2)/k:.1f} km")
a2.plot(x, np.log(I)); a2.set_xlabel("km de aire"); a2.set_ylabel("ln(fracción)"); a2.grid(alpha=0.3)
a2.set_title(f"ln I: recta de pendiente −k = {-k}")
plt.tight_layout(); plt.show()

## 3. La masa de aire

La fórmula plana, $X = 1/\sin h$, se dispara en el horizonte. Una fórmula empírica muy usada, la de Kasten y Young (1989), tiene en cuenta la curvatura de la atmósfera y da unos 38 en el horizonte. No hace falta entenderla: es un ajuste a datos reales. Las comparamos.

In [ ]:
def masa_aire_plana(h_grados):
    return 1 / np.sin(np.radians(h_grados))

def masa_aire_kasten_young(h_grados):
    z = 90 - h_grados                                   # distancia cenital en grados
    return 1 / (np.cos(np.radians(z)) + 0.50572 * (96.07995 - z) ** (-1.6364))

h = np.linspace(0.01, 90, 500)
plt.figure(figsize=(8, 4))
plt.plot(h, masa_aire_plana(h), label="atmósfera plana: 1/sen h")
plt.plot(h, masa_aire_kasten_young(h), label="Kasten y Young (atmósfera curva)")
plt.ylim(0, 45); plt.xlabel("altura sobre el horizonte (°)"); plt.ylabel("masa de aire X")
plt.legend(); plt.grid(alpha=0.3); plt.show()

for hh in [90, 30, 10, 5, 1, 0]:
    print(f"h = {hh:2d}°   plana = {masa_aire_plana(max(hh, 0.01)):8.1f}   curva = {masa_aire_kasten_young(hh):5.1f}")

## 4. Rayleigh: cuánto se apaga cada color

La profundidad óptica de Rayleigh del aire, a nivel del mar y en el cénit, se aproxima bien por

$$\tau(\lambda) \approx 0{,}0088\,\lambda^{-4{,}05}, \quad \lambda \text{ en micrómetros},$$

una fórmula empírica habitual. (El exponente es algo mayor que 4 porque el índice de refracción del aire cambia un poco con el color.) Calculamos la fracción de cada color que llega desde el Sol a distintas alturas. Solo tenemos en cuenta las moléculas: el polvo y la contaminación, que en la realidad añaden bastante extinción, no están en el modelo.

In [ ]:
def tau_rayleigh(lam_nm):
    lam_um = lam_nm / 1000
    return 0.0088 * lam_um ** -4.05

for lam, color in [(450, "azul"), (550, "verde"), (700, "rojo")]:
    print(f"{color:6s} ({lam} nm): τ = {tau_rayleigh(lam):.3f}")

lam = np.linspace(380, 750, 300)
plt.figure(figsize=(8, 4))
for hh in [60, 20, 5, 1]:
    X = masa_aire_kasten_young(hh)
    plt.plot(lam, np.exp(-tau_rayleigh(lam) * X), label=f"Sol a {hh}° (X = {X:.1f})")
plt.xlabel("longitud de onda (nm)"); plt.ylabel("fracción que llega")
plt.legend(); plt.grid(alpha=0.3); plt.title("Transmisión de la atmósfera (solo Rayleigh)")
plt.show()

**Ejercicio.** ¿A qué altura del Sol llega solo el 10 % de la luz azul de 450 nm? Pruébalo con varias alturas o resuélvelo: necesitas $X = \ln 10 / \tau(450)$.

In [ ]:
# Tu código aquí

In [ ]:
# Solución
X_necesaria = np.log(10) / tau_rayleigh(450)
alturas = np.linspace(0, 90, 9001)
h_10 = alturas[np.argmin(np.abs(masa_aire_kasten_young(alturas) - X_necesaria))]
print(f"Masa de aire necesaria: {X_necesaria:.1f}   →   Sol a unos {h_10:.1f}° de altura")

## 5. El color del Sol y del cielo

Ahora traducimos un espectro a un color en la pantalla. Para eso hacen falta dos ingredientes que de momento usamos como cajas negras:

- **El espectro del Sol fuera de la atmósfera.** Lo aproximamos con la curva de Planck de un cuerpo a 5772 K; la entenderás en la semana 31.
- **Cómo ve el color el ojo humano.** Las funciones de igualación de color de la Comisión Internacional de la Iluminación (CIE, 1931) dicen cuánto estimula cada longitud de onda los tres tipos de conos. Usamos una aproximación analítica publicada por Wyman, Sloan y Shirley en 2013, y la fórmula estándar para pasar a colores de pantalla (sRGB).

Los colores que salen son **aproximados**: dependen de la pantalla, y el modelo no incluye el polvo ni la absorción del ozono.

In [ ]:
def planck(lam_nm, T):
    h, c_, k_B = 6.626e-34, 2.998e8, 1.381e-23
    lam = lam_nm * 1e-9
    return 2 * h * c_**2 / lam**5 / (np.exp(h * c_ / (lam * k_B * T)) - 1)

def g(x, mu, s1, s2):
    s = np.where(x < mu, s1, s2)
    return np.exp(-0.5 * ((x - mu) / s) ** 2)

def cie_xyz(lam):
    x = 1.056*g(lam, 599.8, 37.9, 31.0) + 0.362*g(lam, 442.0, 16.0, 26.7) - 0.065*g(lam, 501.1, 20.4, 26.2)
    y = 0.821*g(lam, 568.8, 46.9, 40.5) + 0.286*g(lam, 530.9, 16.3, 31.1)
    z = 1.217*g(lam, 437.0, 11.8, 36.0) + 0.681*g(lam, 459.0, 26.0, 13.8)
    return x, y, z

def espectro_a_rgb(lam, espectro):
    xb, yb, zb = cie_xyz(lam)
    integrar = getattr(np, "trapezoid", None) or np.trapz      # el nombre cambió en NumPy 2
    X, Y, Z = [integrar(espectro * f, lam) for f in (xb, yb, zb)]
    M = np.array([[3.2406, -1.5372, -0.4986], [-0.9689, 1.8758, 0.0415], [0.0557, -0.2040, 1.0570]])
    rgb = M @ np.array([X, Y, Z])
    rgb = np.clip(rgb / rgb.max(), 0, 1)                       # normalizamos el brillo: solo nos interesa el tono
    return np.where(rgb <= 0.0031308, 12.92 * rgb, 1.055 * rgb ** (1 / 2.4) - 0.055)

lam = np.linspace(380, 750, 400)
sol = planck(lam, 5772)
alturas = [60, 30, 15, 8, 4, 2, 1, 0.5]

fig, ax = plt.subplots(figsize=(10, 2.6))
for i, hh in enumerate(alturas):
    color = espectro_a_rgb(lam, sol * np.exp(-tau_rayleigh(lam) * masa_aire_kasten_young(hh)))
    ax.add_patch(plt.Circle((i, 0), 0.4, color=color))
    ax.text(i, -0.65, f"{hh}°", ha="center", color="white")
cielo = espectro_a_rgb(lam, sol * tau_rayleigh(lam))     # luz dispersada: proporcional a τ(λ)
ax.add_patch(plt.Rectangle((len(alturas) - 0.4, -0.4), 1.2, 0.8, color=cielo))
ax.text(len(alturas) + 0.2, -0.65, "cielo", ha="center", color="white")
ax.set_xlim(-0.6, len(alturas) + 1); ax.set_ylim(-0.9, 0.6); ax.set_aspect("equal"); ax.axis("off")
ax.set_facecolor("black"); fig.patch.set_facecolor("#222222")
plt.title("Color aproximado del Sol según su altura, y del cielo", color="white"); plt.show()

Con el Sol alto, el color sale casi blanco (ligeramente amarillento por el modelo); al bajar se vuelve amarillo, naranja y rojo. El rectángulo de la derecha es la luz dispersada por las moléculas, proporcional a $\tau(\lambda)$: sale azul.

## 6. Experimenta

1. Cambia la temperatura del Sol a 3500 K (una estrella roja) y a 10 000 K (una estrella azul) y mira cómo cambian los colores.
2. Multiplica la profundidad óptica por 3, como si hubiera mucho polvo fino en el aire. ¿Qué pasa con el color del Sol a 15°?
3. Haz que la dispersión no dependa del color (cambia el exponente $-4{,}05$ por 0), como en una nube. ¿De qué color sale el cielo?

In [ ]:
# Tu experimento aquí